# Total HFI skies: CMB + tSZ + CIB + homogeneous white noise

Build six Planck HFI frequency maps

$$\mathrm{total}(\nu) = b_\nu * \big(\mathrm{CMB} + \mathrm{tSZ}(\nu) + \mathrm{CIB}(\nu)\big) + n_\nu^\mathrm{homog}$$

and write them to

`/rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/`

**No kSZ.** The Gaussian beam $b_\nu$ is Table I FWHM, applied to the **signal only**. Homogeneous noise is the pixel-white realisation from `homogeneous_planck_white_noise.ipynb` (not beam-smoothed).

**CIB–tSZ correlation.** Both come from the official FLAMINGO `lightcone0` **lensed, co-rotated** maps (`lensed_tSZ_rot.fits` and `lensed_CIB_rot_BANDPASS_F*` / SED-scaled 100 and 143 GHz). The same shell rotations keep the physical CIB–tSZ cross-correlation; this notebook does not redraw either field.

| | |
|---|---|
| Frequencies | 100, 143, 217, 353, 545, 857 GHz |
| $N_\mathrm{side}$ | 2048 (components downgraded from 4096) |
| Units | $\mu\mathrm{K}_\mathrm{CMB}$ |
| Beam | Gaussian Table I FWHM on CMB+tSZ+CIB; noise unsmoothed |
| Pixel window | **not** deconvolved — leave as observed for MMF / further processing (`PIXWIN=0` in FITS). Yang-style $W_\ell^\mathrm{pix}(4096)$ applies only to native FLAMINGO $N_\mathrm{side}=4096$ signal $C_\ell$. |


## 1. Configuration

In [1]:
from pathlib import Path

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from flamingo_mock.config import BEAM_FWHM_ARCMIN, PLANCK_FREQUENCIES_GHZ
from flamingo_mock.io import write_map
from flamingo_mock.powerspectra import bin_cl, compute_cl

NSIDE = 2048
NSIDE_IN = 4096
APPLY_BEAM = True
WRITE_MAPS = True
OVERWRITE = False
LMAX_CL = 2 * NSIDE
DELTA_ELL = 21

FREQS = tuple(int(f) for f in PLANCK_FREQUENCIES_GHZ)

COMP = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic/components")
NOISE = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic/planck_noise/homogeneous")
OUT = Path("/rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test")
FIG_DIR = Path("../figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

print("freqs :", FREQS)
print("nside :", NSIDE, "(from", NSIDE_IN, ")")
print("beam  :", APPLY_BEAM, BEAM_FWHM_ARCMIN)
print("out   :", OUT)


freqs : (100, 143, 217, 353, 545, 857)
nside : 2048 (from 4096 )
beam  : True {100: 9.66, 143: 7.22, 217: 4.9, 353: 4.92, 545: 4.67, 857: 4.22}
out   : /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test


## 2. Paths

In [2]:
def cmb_path():
    return COMP / "cmb" / "primary_CMB_T_lensed_nside4096_seed42.fits"

def tsz_path(nu):
    return COMP / "tsz" / f"tSZ_deltaT_{nu}GHz_nside{NSIDE_IN}.fits"

def cib_path(nu):
    return COMP / "cib" / f"CIB_deltaT_{nu}GHz_nside{NSIDE_IN}.fits"

def y_path():
    return COMP / "tsz" / f"compton_y_nside{NSIDE_IN}.fits"

def noise_path(nu):
    return NOISE / f"{nu}GHz" / f"white_noise_{nu}GHz_nside{NSIDE}_uK.fits"

def total_path(nu):
    return OUT / f"sky_CMB_tSZ_CIB_homog_{nu}GHz_nside{NSIDE}_uK.fits"


missing = []
for nu in FREQS:
    for p in (tsz_path(nu), cib_path(nu), noise_path(nu)):
        if not p.is_file():
            missing.append(p)
if not cmb_path().is_file():
    missing.append(cmb_path())
if not y_path().is_file():
    missing.append(y_path())
if missing:
    raise FileNotFoundError("missing inputs:\n  " + "\n  ".join(str(p) for p in missing))

for nu in FREQS:
    print(f"{nu:3d}  tSZ {tsz_path(nu).name}")
    print(f"     CIB {cib_path(nu).name}")
    print(f"     noi {noise_path(nu).name}")
print("CMB ", cmb_path().name)


100  tSZ tSZ_deltaT_100GHz_nside4096.fits
     CIB CIB_deltaT_100GHz_nside4096.fits
     noi white_noise_100GHz_nside2048_uK.fits
143  tSZ tSZ_deltaT_143GHz_nside4096.fits
     CIB CIB_deltaT_143GHz_nside4096.fits
     noi white_noise_143GHz_nside2048_uK.fits
217  tSZ tSZ_deltaT_217GHz_nside4096.fits
     CIB CIB_deltaT_217GHz_nside4096.fits
     noi white_noise_217GHz_nside2048_uK.fits
353  tSZ tSZ_deltaT_353GHz_nside4096.fits
     CIB CIB_deltaT_353GHz_nside4096.fits
     noi white_noise_353GHz_nside2048_uK.fits
545  tSZ tSZ_deltaT_545GHz_nside4096.fits
     CIB CIB_deltaT_545GHz_nside4096.fits
     noi white_noise_545GHz_nside2048_uK.fits
857  tSZ tSZ_deltaT_857GHz_nside4096.fits
     CIB CIB_deltaT_857GHz_nside4096.fits
     noi white_noise_857GHz_nside2048_uK.fits
CMB  primary_CMB_T_lensed_nside4096_seed42.fits


## 3. Load, downgrade, beam-smooth, add noise

`hp.ud_grade` $4096\to 2048$ on each component, sum, Gaussian smooth, then add the homogeneous map.
CMB is frequency-independent and is downgraded once.


In [3]:
def load_uK(path):
    return np.asarray(hp.read_map(str(path), field=0, dtype=np.float64))


def to_nside(m, nside=NSIDE):
    if hp.get_nside(m) == nside:
        return m
    return hp.ud_grade(m, nside)


print("CMB: load + ud_grade ...", flush=True)
cmb = to_nside(load_uK(cmb_path()))
print(f"  nside={hp.get_nside(cmb)}  rms={cmb.std():.2f} uK")

stats = {}
written, skipped = [], []
for nu in FREQS:
    dest = total_path(nu)
    print(f"\n=== {nu} GHz ===", flush=True)
    tsz = to_nside(load_uK(tsz_path(nu)))
    cib = to_nside(load_uK(cib_path(nu)))
    noise = load_uK(noise_path(nu))
    if hp.get_nside(noise) != NSIDE:
        raise ValueError(f"noise nside {hp.get_nside(noise)} != {NSIDE}")

    signal = cmb + tsz + cib
    fwhm_arcmin = float(BEAM_FWHM_ARCMIN[nu])
    if APPLY_BEAM:
        print(f"  smoothing FWHM={fwhm_arcmin:.2f}' ...", flush=True)
        signal_b = hp.smoothing(signal, fwhm=np.radians(fwhm_arcmin / 60.0))
    else:
        signal_b = signal
    total = signal_b + noise

    stats[nu] = dict(
        cmb=float(cmb.std()),
        tsz=float(tsz.std()),
        cib=float(cib.std()),
        signal=float(signal.std()),
        signal_beam=float(signal_b.std()),
        noise=float(noise.std()),
        total=float(total.std()),
        fwhm=fwhm_arcmin,
    )
    print(
        f"  rms uK  CMB={cmb.std():.2f}  tSZ={tsz.std():.2f}  CIB={cib.std():.2f}  "
        f"sig={signal.std():.2f}  beamed={signal_b.std():.2f}  "
        f"noise={noise.std():.2f}  total={total.std():.2f}"
    )

    if dest.is_file() and not OVERWRITE:
        skipped.append(nu)
        print(f"  skip {dest}")
    elif WRITE_MAPS:
        extra = [
            ("COMPS", "CMB+tSZ+CIB+homog_noise"),
            ("FWHM", fwhm_arcmin, "Gaussian beam [arcmin] on signal only"),
            ("BEAMON", int(APPLY_BEAM)),
            ("NOISE", "homogeneous white, not beamed"),
            ("NKSZ", 1, "kSZ not included"),
            ("PIXWIN", 0, "0=do NOT deconvolve HEALPix pixwin"),
        ]
        write_map(dest, total, unit="uK_CMB", freq=float(nu), extra=extra, dtype=np.float32)
        written.append(nu)
    del tsz, cib, noise, signal, signal_b, total

print(f"\nwrote {len(written)}  skipped {len(skipped)}")
print(f"{'nu':>5}  {'CMB':>8}  {'tSZ':>8}  {'CIB':>8}  {'beamed':>8}  {'noise':>8}  {'total':>8}")
for nu in FREQS:
    s = stats[nu]
    print(f"{nu:5d}  {s['cmb']:8.2f}  {s['tsz']:8.2f}  {s['cib']:8.2f}  "
          f"{s['signal_beam']:8.2f}  {s['noise']:8.2f}  {s['total']:8.2f}")


CMB: load + ud_grade ...


  nside=2048  rms=107.55 uK

=== 100 GHz ===


  smoothing FWHM=9.66' ...


  rms uK  CMB=107.55  tSZ=7.69  CIB=2.81  sig=107.84  beamed=99.75  noise=45.06  total=109.45
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_100GHz_nside2048_uK.fits (0.20 GB)

=== 143 GHz ===


  smoothing FWHM=7.22' ...


...

  rms uK  CMB=107.55  tSZ=11.43  CIB=117.12  sig=160.74  beamed=119.58  noise=89.67  total=149.46
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_353GHz_nside2048_uK.fits (0.20 GB)

=== 545 GHz ===


  smoothing FWHM=4.67' ...


  rms uK  CMB=107.55  tSZ=28.56  CIB=1617.53  sig=1626.77  beamed=800.36  noise=469.61  total=928.01
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_545GHz_nside2048_uK.fits (0.20 GB)

=== 857 GHz ===


  smoothing FWHM=4.22' ...


  rms uK  CMB=107.55  tSZ=56.57  CIB=136114.62  sig=136127.71  beamed=65675.46  noise=11125.45  total=66610.31
  wrote /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_857GHz_nside2048_uK.fits (0.20 GB)

wrote 6  skipped 0
   nu       CMB       tSZ       CIB    beamed     noise     total
  100    107.55      7.69      2.81     99.75     45.06    109.45
  143    107.55      5.31      6.03    102.33     19.21    104.12
  217    107.55      0.04     18.89    105.04     27.24    108.53
  353    107.55     11.43    117.12    119.58     89.67    149.46
  545    107.55     28.56   1617.53    800.36    469.61    928.01
  857    107.55     56.57  136114.62  65675.46  11125.45  66610.31


## 4. CIB–tSZ correlation check (353 GHz)

353 GHz is an official released CIB band and a strong tSZ channel. The cross-spectrum of CIB $\Delta T$ with Compton $y$ (same FLAMINGO rotations) should be clearly non-zero. Noise and the CMB are uncorrelated with $y$ at high $\ell$.


In [4]:
nu_x = 353
print("load 353 GHz CIB, tSZ, y, total ...", flush=True)
cib353 = to_nside(load_uK(cib_path(nu_x)))
tsz353 = to_nside(load_uK(tsz_path(nu_x)))
y = to_nside(load_uK(y_path()))
tot353 = load_uK(total_path(nu_x))

def pearson(a, b):
    a = a - a.mean()
    b = b - b.mean()
    return float(np.dot(a, b) / np.sqrt(np.dot(a, a) * np.dot(b, b)))

print(f"pixel r(CIB, tSZ)  = {pearson(cib353, tsz353):.4f}")
print(f"pixel r(CIB, y)    = {pearson(cib353, y):.4f}")
print(f"pixel r(tSZ, y)    = {pearson(tsz353, y):.4f}   (should be -1: tSZ = y * f(nu) < 0)")
print(f"pixel r(total, y)  = {pearson(tot353, y):.4f}")

print("cross-spectra ...", flush=True)
cl_cib_y = compute_cl(cib353, y, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
cl_cib = compute_cl(cib353, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
cl_y = compute_cl(y, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
ell_b, cx = bin_cl(cl_cib_y, delta_ell=DELTA_ELL, lmin=2)
_, cc = bin_cl(cl_cib, delta_ell=DELTA_ELL, lmin=2)
_, cy = bin_cl(cl_y, delta_ell=DELTA_ELL, lmin=2)
rho = cx / np.sqrt(cc * cy)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].semilogx(ell_b, ell_b * (ell_b + 1) * cx / (2 * np.pi), color="C1", lw=1.6)
axes[0].set_xlabel(r"Multipole $\ell$")
axes[0].set_ylabel(r"$\ell(\ell+1)C_\ell^{\mathrm{CIB}\times y}/2\pi$")
axes[0].set_title("353 GHz CIB $\times$ Compton $y$")
axes[0].set_xlim(2, LMAX_CL)
axes[1].semilogx(ell_b, rho, color="C0", lw=1.6)
axes[1].axhline(0.0, color="k", lw=0.8)
axes[1].set_xlabel(r"Multipole $\ell$")
axes[1].set_ylabel(r"$C_\ell^{\mathrm{CIB}\times y}/\sqrt{C_\ell^{\mathrm{CIB}}C_\ell^{y}}$")
axes[1].set_title("CIB–$y$ correlation coefficient")
axes[1].set_xlim(2, LMAX_CL)
axes[1].set_ylim(-1.05, 1.05)
fig.tight_layout()
out = FIG_DIR / "total_maps_cib_tsz_cross_353GHz.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
m = (ell_b > 50) & (ell_b < 500) & np.isfinite(rho)
print(f"mean rho (50 < ell < 500) = {np.nanmean(rho[m]):.3f}")
plt.show()


load 353 GHz CIB, tSZ, y, total ...


pixel r(CIB, tSZ)  = 0.1563
pixel r(CIB, y)    = 0.1563


pixel r(tSZ, y)    = 1.0000   (should be -1: tSZ = y * f(nu) < 0)
pixel r(total, y)  = 0.0891
cross-spectra ...


wrote ../figures/total_maps_cib_tsz_cross_353GHz.png
mean rho (50 < ell < 500) = 0.144


## 5. Power spectrum at 143 GHz: signal, noise, total

In [5]:
nu = 143
tot = load_uK(total_path(nu))
noi = load_uK(noise_path(nu))
sig = tot - noi   # beamed CMB+tSZ+CIB

cl_tot = compute_cl(tot, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
cl_noi = compute_cl(noi, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
cl_sig = compute_cl(sig, lmax=LMAX_CL, iter=0, deconv_pixel_window=False)
et, ct = bin_cl(cl_tot, delta_ell=DELTA_ELL, lmin=2)
en, cn = bin_cl(cl_noi, delta_ell=DELTA_ELL, lmin=2)
es, cs = bin_cl(cl_sig, delta_ell=DELTA_ELL, lmin=2)

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.loglog(es, cs, color="C2", lw=1.6, label="CMB+tSZ+CIB (beamed)")
ax.loglog(en, cn, color="0.45", lw=1.4, ls="--", label="homogeneous noise")
ax.loglog(et, ct, color="C0", lw=1.6, label="total")
ax.set_xlabel(r"Multipole $\ell$")
ax.set_ylabel(r"$C_\ell$ [$\mu\mathrm{K}^2$]")
ax.set_title(f"{nu} GHz")
ax.set_xlim(2, LMAX_CL)
ax.legend(frameon=False)
fig.tight_layout()
out = FIG_DIR / f"total_maps_cl_{nu}GHz.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
print("wrote", out)
plt.show()


wrote ../figures/total_maps_cl_143GHz.png


## 6. Mollview gallery

In [6]:
fig = plt.figure(figsize=(12.5, 7.2))
for i, nu in enumerate(FREQS, 1):
    m = load_uK(total_path(nu))
    lo, hi = np.percentile(m, [1, 99])
    hp.mollview(
        m, fig=fig.number, sub=(2, 3, i),
        min=lo, max=hi, cmap="RdBu_r", notext=True,
        title=f"{nu} GHz  rms={m.std():.1f} uK",
        unit=r"$\mu\mathrm{K}$",
    )
fig.suptitle("CMB + tSZ + CIB (beamed) + homogeneous white noise", y=1.02)
out = FIG_DIR / "total_maps_hfi_homog_mollview.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("wrote", out)
plt.show()

# 353 GHz components on a common scale (downgraded, no noise)
fig = plt.figure(figsize=(12.5, 4.4))
vmax = np.percentile(np.abs(np.concatenate([cib353, tsz353])), 99)
hp.mollview(cib353, fig=fig.number, sub=(1, 3, 1), min=-vmax, max=vmax,
            cmap="RdBu_r", notext=True, title="CIB 353 GHz", unit=r"$\mu\mathrm{K}$")
hp.mollview(tsz353, fig=fig.number, sub=(1, 3, 2), min=-vmax, max=vmax,
            cmap="RdBu_r", notext=True, title="tSZ 353 GHz", unit=r"$\mu\mathrm{K}$")
hp.mollview(tot353, fig=fig.number, sub=(1, 3, 3),
            min=np.percentile(tot353, 1), max=np.percentile(tot353, 99),
            cmap="RdBu_r", notext=True, title="total 353 GHz", unit=r"$\mu\mathrm{K}$")
out = FIG_DIR / "total_maps_353GHz_cib_tsz_total.png"
fig.savefig(out, dpi=120, bbox_inches="tight")
print("wrote", out)
plt.show()


wrote ../figures/total_maps_hfi_homog_mollview.png


wrote ../figures/total_maps_353GHz_cib_tsz_total.png


## 7. Inventory

In [7]:
print("Total maps")
for nu in FREQS:
    p = total_path(nu)
    if p.is_file():
        print(f"  {nu:3d} GHz  {p.stat().st_size/1e6:7.1f} MB  {p}")
    else:
        print(f"  {nu:3d} GHz  MISSING  {p}")
print()
print("figures:")
for name in (
    "total_maps_cib_tsz_cross_353GHz.png",
    "total_maps_cl_143GHz.png",
    "total_maps_hfi_homog_mollview.png",
    "total_maps_353GHz_cib_tsz_total.png",
):
    p = FIG_DIR / name
    print(f"  {'ok' if p.is_file() else 'MISSING'}  {p.resolve()}")


Total maps
  100 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_100GHz_nside2048_uK.fits
  143 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_143GHz_nside2048_uK.fits
  217 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_217GHz_nside2048_uK.fits
  353 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_353GHz_nside2048_uK.fits
  545 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_545GHz_nside2048_uK.fits
  857 GHz    201.3 MB  /rds/rds-lxu/flamingo/integrated_maps_synthetic/total_maps/test/sky_CMB_tSZ_CIB_homog_857GHz_nside2048_uK.fits

figures:
  ok  /scratch/scratch-lxu/flamingo_mock_analysis/figures/total_maps_cib_tsz_cross_353GHz.png
  ok  /scratch/scratch-lxu/flamingo_mock_analysis/figures/total_maps_cl_143GHz.pn